# 업태·종목 기반 자금종류 추천 시스템 (Qwen3-Embedding + BM25 + Qdrant)

이 노트북은 Qwen3-Embedding-0.6B와 BM25, Qdrant를 사용하여 업태와 종목을 입력하면 적합한 자금종류를 추천하는 시스템입니다.

**핵심 기능:**
- **Dense vectors**: Qwen3-Embedding-0.6B로 의미론적 유사도 측정 (1024차원)
- **Sparse vectors**: BM25로 키워드 매칭
- **Hybrid Search**: Dense + Sparse 결과 결합

## Step 1: Setup for Google Colab

In Google Colab, we'll use Qdrant in-memory mode instead of Docker.

In [ ]:
# No Docker needed for Colab! Qdrant will run in-memory mode.
print("✓ Ready to use Qdrant in-memory mode")

## Step 2: Install Required Python Packages

In [ ]:
%pip install -U transformers
%pip install -U torch
%pip install pandas
%pip install qdrant_client
%pip install rank-bm25
%pip install tqdm
%pip install ipywidgets

## Step 3: Import Required Libraries

In [ ]:
import pandas as pd
import json
import torch
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModel
from qdrant_client import QdrantClient, models
from rank_bm25 import BM25Okapi
import numpy as np

## Step 4: 자금 데이터셋 로드 (업태, 종목, 자금종류)

In [ ]:
# Load funding data from CSV file
def load_funding_data(file_path='funding_data.csv'):
    funding_df = pd.read_csv(file_path, sep='|')
    funding_json = funding_df.to_dict(orient='records')
    
    # Print the first record as JSON
    print(json.dumps(funding_json[0], indent=2, ensure_ascii=False))
    print(f"Total records: {len(funding_json)}")
    
    return funding_df, funding_json

funding_df, funding_json = load_funding_data()

## Step 5: Initialize Qwen3-Embedding-0.6B Model

In [ ]:
def initialize_qwen3_model():
    """Initialize Qwen3-Embedding-0.6B model"""
    model_name = "Qwen/Qwen3-Embedding-0.6B"
    
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
    
    # Move to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    
    print(f"✓ Model loaded on {device}")
    return tokenizer, model, device

tokenizer, model, device = initialize_qwen3_model()

## Step 6: Initialize BM25 Index

In [ ]:
def create_business_text(record):
    """Format business information for embedding and BM25"""
    return f"업태: {record['업태']}\n종목: {record['종목']}"

def tokenize_korean(text):
    """Simple Korean tokenizer for BM25 (character-based)"""
    # Remove special characters and split by space
    import re
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    return text.split()

# Create BM25 index
print("Creating BM25 index...")
corpus_texts = [create_business_text(record) for record in funding_json]
tokenized_corpus = [tokenize_korean(text) for text in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

print(f"✓ BM25 index created with {len(corpus_texts)} documents")

## Step 7: 업태·종목 텍스트 포맷팅 및 Dense 임베딩 생성

In [ ]:
def generate_dense_embedding(text, tokenizer, model, device):
    """Generate dense embedding using Qwen3"""
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate embedding
    with torch.no_grad():
        outputs = model(**inputs)
        # Use mean pooling
        embeddings = outputs.last_hidden_state.mean(dim=1)
        # Normalize
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    
    return embeddings[0].cpu().numpy()

# Test with sample record
sample_record = funding_df.iloc[0]
business_text = create_business_text(sample_record)

print("\n포맷된 업태·종목 텍스트:")
print(business_text)
print(f"\n해당하는 자금종류: {sample_record['자금종류']}")

# Generate test embedding
test_embedding = generate_dense_embedding(business_text, tokenizer, model, device)
print(f"\n✓ Dense embedding shape: {test_embedding.shape}")
print(f"First 5 elements: {test_embedding[:5]}")

## Step 8: 모든 업태·종목에 대한 Dense 임베딩 생성

In [ ]:
def generate_all_embeddings(records, tokenizer, model, device):
    """Generate dense embeddings for all records"""
    all_embeddings = []
    
    for record in tqdm(records, desc="Generating embeddings"):
        business_text = create_business_text(record)
        embedding = generate_dense_embedding(business_text, tokenizer, model, device)
        
        all_embeddings.append({
            "record": record,
            "dense_vector": embedding
        })
    
    print(f"✓ Generated embeddings for {len(all_embeddings)} records")
    return all_embeddings

all_embeddings = generate_all_embeddings(funding_json, tokenizer, model, device)

## Step 9: Qdrant 컬렉션 생성 (Dense Only)

In [ ]:
def create_qdrant_collection(collection_name="funding_qwen3", vector_size=1024):
    """Create Qdrant collection for dense vectors only"""
    client = QdrantClient(":memory:")  # In-memory mode for Colab
    
    # Create collection with dense vectors only
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE
        )
    )
    
    print(f"✓ Collection '{collection_name}' created successfully in-memory")
    return client

client = create_qdrant_collection(vector_size=test_embedding.shape[0])

## Step 10: Qdrant 컬렉션에 자금 데이터 삽입

In [ ]:
def insert_to_qdrant(client, embeddings, collection_name="funding_qwen3"):
    """Insert embeddings into Qdrant collection"""
    for embedding_data in tqdm(embeddings, desc="Inserting to Qdrant"):
        record = embedding_data["record"]
        dense_vector = embedding_data["dense_vector"]
        
        client.upsert(
            collection_name=collection_name,
            points=[
                models.PointStruct(
                    id=record["Id"],
                    payload=record,
                    vector=dense_vector.tolist()
                )
            ]
        )
    
    print(f"✓ Successfully inserted {len(embeddings)} records")

insert_to_qdrant(client, all_embeddings)

## Step 11: 하이브리드 검색 함수 (BM25 + Dense)

In [ ]:
def search_funding_types_hybrid(client, tokenizer, model, device, bm25, funding_json,
                                business_type, business_item, limit=10, threshold=0.8,
                                collection_name="funding_qwen3"):
    """Hybrid search using BM25 + Dense vectors
    
    Args:
        threshold: 최소 점수 임계치 (0.0 ~ 1.0, 기본값 0.8)
    """
    search_query = f"업태: {business_type}\n종목: {business_item}"
    
    # 1. BM25 search
    tokenized_query = tokenize_korean(search_query)
    bm25_scores = bm25.get_scores(tokenized_query)
    
    # Normalize BM25 scores to 0-1
    max_bm25_score = max(bm25_scores) if max(bm25_scores) > 0 else 1
    bm25_scores_norm = bm25_scores / max_bm25_score
    
    # Get top BM25 results
    bm25_top_indices = np.argsort(bm25_scores_norm)[::-1][:limit * 2]
    
    # 2. Dense search with Qdrant
    query_embedding = generate_dense_embedding(search_query, tokenizer, model, device)
    dense_results = client.query_points(
        collection_name,
        query=query_embedding.tolist(),
        limit=limit * 2
    )
    
    # 3. Combine results
    combined = {}
    
    # Add BM25 results
    for idx in bm25_top_indices:
        score = float(bm25_scores_norm[idx])
        if score >= threshold:
            record_id = funding_json[idx]["Id"]
            combined[record_id] = {
                "record": funding_json[idx],
                "score": score,
                "source": "BM25"
            }
    
    # Add Dense results (keep higher score)
    for point in dense_results.points:
        if point.score >= threshold:
            if point.id in combined:
                if point.score > combined[point.id]["score"]:
                    combined[point.id] = {
                        "record": point.payload,
                        "score": point.score,
                        "source": "Dense"
                    }
            else:
                combined[point.id] = {
                    "record": point.payload,
                    "score": point.score,
                    "source": "Dense"
                }
    
    # Sort by score
    sorted_results = sorted(combined.values(), key=lambda x: x["score"], reverse=True)[:limit]
    
    # Create result object
    class SearchResults:
        def __init__(self, results):
            self.points = []
            for r in results:
                class Point:
                    def __init__(self, record, score):
                        self.payload = record
                        self.score = score
                        self.id = record["Id"]
                self.points.append(Point(r["record"], r["score"]))
    
    return SearchResults(sorted_results)

## Step 12: 추천된 자금종류 결과 표시

In [ ]:
def display_funding_results(results, business_type, business_item):
    """Display funding type recommendations in a readable format"""
    print(f"입력: 업태='{business_type}', 종목='{business_item}'")
    print("=" * 60)
    print(f"\n추천 자금종류 ({len(results.points)}개):\n")
    
    # Collect all funding types
    funding_types_set = set()
    
    for i, result in enumerate(results.points):    
        record = result.payload
        print(f"{i+1}. 유사 업태/종목: {record['업태']} - {record['종목']}")
        print(f"   매칭 점수: {result.score:.2f}")
        print(f"   자금종류: {record['자금종류']}")
        
        # Add funding type to set (each row has single funding type)
        funding_types_set.add(record['자금종류'])
        print()
    
    print("=" * 60)
    print(f"전체 추천 자금종류 키워드 ({len(funding_types_set)}개):")
    print(", ".join(sorted(funding_types_set)))

## Step 13: 예제 검색 테스트

In [ ]:
# 예제 1: 제조업 - 스마트폰 제조
result = search_funding_types_hybrid(client, tokenizer, model, device, bm25, funding_json,
                                     business_type="제조업", 
                                     business_item="스마트폰 제조", 
                                     limit=5, 
                                     threshold=0.8)
display_funding_results(result, "제조업", "스마트폰 제조")

In [ ]:
# 예제 2: 서비스업 - 한식 레스토랑
result = search_funding_types_hybrid(client, tokenizer, model, device, bm25, funding_json,
                                     business_type="서비스업", 
                                     business_item="한식 레스토랑", 
                                     limit=5, 
                                     threshold=0.8)
display_funding_results(result, "서비스업", "한식 레스토랑")

In [ ]:
# 예제 3: IT업 - 클라우드 플랫폼 개발
result = search_funding_types_hybrid(client, tokenizer, model, device, bm25, funding_json,
                                     business_type="IT업", 
                                     business_item="클라우드 플랫폼 개발", 
                                     limit=5, 
                                     threshold=0.8)
display_funding_results(result, "IT업", "클라우드 플랫폼 개발")

In [ ]:
# 예제 4: 농림어업 - 스마트팜 운영
result = search_funding_types_hybrid(client, tokenizer, model, device, bm25, funding_json,
                                     business_type="농림어업", 
                                     business_item="스마트팜 운영", 
                                     limit=5, 
                                     threshold=0.8)
display_funding_results(result, "농림어업", "스마트팜 운영")

## Step 14: 대화형 테스트 (업태·종목 직접 입력)

In [ ]:
# 대화형 테스트: 업태와 종목을 직접 입력하여 자금종류 추천받기 (반복 실행)
print("=" * 60)
print("업태·종목 기반 자금종류 추천 시스템 (Qwen3 + BM25)")
print("=" * 60)
print("종료하려면 업태 입력 시 'q' 또는 '종료'를 입력하세요.")
print("=" * 60)

while True:
    print()
    
    # 업태 입력
    business_type_input = input("업태를 입력하세요 (예: 제조업, 서비스업, IT업 등): ").strip()
    
    # 종료 조건 체크
    if business_type_input.lower() in ['q', 'quit', 'exit', '종료', '끝']:
        print("\n프로그램을 종료합니다. 감사합니다!")
        break
    
    if not business_type_input:
        print("⚠️ 업태를 입력해주세요.")
        continue
    
    # 종목 입력
    business_item_input = input("종목을 입력하세요 (예: 전자제품 제조, 카페, 소프트웨어 개발 등): ").strip()
    
    if not business_item_input:
        print("⚠️ 종목을 입력해주세요.")
        continue
    
    # 점수 임계치 입력 (선택사항)
    threshold_input = input("점수 임계치를 입력하세요 (0.0~1.0, 기본값 0.8, Enter로 기본값 사용): ").strip()
    if threshold_input:
        try:
            threshold_value = float(threshold_input)
            # 0.0 ~ 1.0 범위로 제한
            threshold_value = max(0.0, min(1.0, threshold_value))
        except ValueError:
            print(f"⚠️ 잘못된 입력값입니다. 기본값 0.8을 사용합니다.")
            threshold_value = 0.8
    else:
        threshold_value = 0.8
    
    print()
    print(f"검색 중... (임계치: {threshold_value})")
    print()
    
    try:
        # 검색 수행
        result = search_funding_types_hybrid(client, tokenizer, model, device, bm25, funding_json,
                                            business_type=business_type_input, 
                                            business_item=business_item_input,
                                            limit=10,
                                            threshold=threshold_value)
        
        # 결과 표시
        display_funding_results(result, business_type_input, business_item_input)
        
    except Exception as e:
        print(f"❌ 검색 중 오류가 발생했습니다: {e}")
        print("다시 시도해주세요.")
    
    print("\n" + "-" * 60)